## How to run this notebook

There are two ways. Both are safe; choose either.

**Way 1: run your own copy.** No warning, and your changes are kept.

- At the top of this page, choose **Copy to Drive** (or **File → Save a copy in Drive**).
- Your copy opens in a new tab. It is saved in your Google Drive, in the folder **Colab Notebooks**.
- In your copy, choose **Runtime → Run all**.

**Way 2: run this page directly.** Quicker, but your changes are not kept.

- Choose **Runtime → Run all**.
- Colab shows this warning:

  > Warning: This notebook was not authored by Google. This notebook is being loaded from GitHub. It may request access to your data stored with Google, or read data and credentials from other sessions. Please review the source code before executing this notebook.

- Choose **Run anyway**.

**Why it is safe.** Colab shows this warning for every notebook that does not come from Google, before it has looked at the code. This notebook does not ask for your Google account, your Drive or any other data of yours. It only saves the table it analyses, and any figures it draws, in this session, which ends when you close the page. You can read every line of the code below before you run it.

# Bakery flour orders: autocorrelation of a zigzag

**Autocorrelation** · a worked example from the StatNex teaching guide

**The story.** A bakery records how much flour it orders each day. A big order one day leaves flour on the shelf, so the next order is small, and then big again.

**The question.** Is there a predictable pattern from one day's order to the next?

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `08_time_series/autocorrelation.csv` (10 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
Day,Flour ordered (kg)
1.0,220.0
2.0,158.0
3.0,233.0
4.0,149.0
5.0,241.0
6.0,172.0
7.0,226.0
8.0,151.0
9.0,238.0
10.0,166.0
""")

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import acf, levinson_durbin

pd.set_option("display.width", 120)

## The data

In [ ]:
# One time series in row order. The order of the rows is the data, so it may not have
# blank cells.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
x = data["Flour ordered (kg)"].to_numpy(float)
n = len(x)
observations = pd.DataFrame({"No.": np.arange(1, n + 1), "Flour ordered (kg)": x})
print(observations.to_string(index=False))

## Autocorrelation

In [ ]:
# Is each value like the one k periods earlier? r(k) is the correlation of the series
# with itself k periods back, for every lag up to n - 2: near -1, a high value is
# followed by a low one (a zigzag); near +1, by another high one. Each lag's sum of
# cross-products is divided by the same total sum of squares, the standard estimator
# (R's acf()), so r(k) shrinks a little as k grows (fft=False computes it directly).
max_lag = n - 2
lags = np.arange(1, max_lag + 1)
r = acf(x, nlags=max_lag, fft=False)[1:]
autocorrelation = pd.DataFrame({"Lag": lags, "Autocorrelation": r})
print()
print(autocorrelation.to_string(index=False))
# A rough guide for one lag: if the values were independent, r(k) would fall within
# +/- 1.96 / sqrt(n) about 95% of the time. With few values the guide is rough, and
# checking many lags makes one stray value likely.
print(f"\nRough 95% band for one lag: +/- {1.96 / np.sqrt(n):.4f}")

## Box-Ljung test

In [ ]:
# One test of lags 1 to k together. Q(k) = n (n + 2) * sum of r(j)^2 / (n - j) over
# j = 1..k, compared with chi-square on k degrees of freedom. H0: "the autocorrelations
# up to lag k are all zero" (no pattern from one period to the next). A small P says
# some pattern is there, not which lag carries it. The chi-square is a large-sample
# approximation, rough for a short series.
ljung = acorr_ljungbox(x, lags=max_lag)
print()
print(ljung)

## Partial autocorrelation

In [ ]:
# What is left of the lag-k correlation once lags 1..k-1 are accounted for, by the
# Durbin-Levinson recursion on the autocorrelations above (the Yule-Walker estimate,
# as R's pacf()). A large value at lag 1 and small ones after it point to a series
# driven by its last value alone. statsmodels' pacf() stops at half the series;
# levinson_durbin() is the same recursion without that limit.
partial = levinson_durbin(x, nlags=max_lag, isacov=False)[2][1:]
partial_table = pd.DataFrame({"Lag": lags, "Partial autocorrelation": partial})
print()
print(partial_table.to_string(index=False))

## What the result says

Autocorrelation measures how strongly a series is linked to its own past. The lag is how many days back we look. A coefficient near 1 means the two days move together, and a coefficient near -1 means they move in opposite ways.

At lag 1 the autocorrelation coefficient is -0.90: a big order is followed by a small one. At lag 2 it is 0.76: orders two days apart are alike. The signs keep alternating at later lags, which is the mark of a zigzag. The Box-Ljung statistic at lag 1 is 10.92 with a P value of 0.001. The P value is the chance of a link this strong in a series with no pattern.

So today's order almost tells you tomorrow's. But 10 days are a short record, and orders are not the same as the flour the bakery uses.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.